# BP1 Gate 1 — Business Understanding & Policy
**Customer360 Navigator Enterprise Suite — Customer Intent Classification**

## Why this notebook exists, and why it runs before Gate 3
Master Execution Plan Section 8 (6-Gate Governance SOP) is explicit: "Every one of the 8 Business
Problems moves through the same six gates, **in order**." BP1's Gate 2 notebook
(`bp1_customer_intent_classification_g2_data_integration_taxonomy_mapping.ipynb`) was built and real-run
confirmed first — that sequencing gap is closed here, retroactively, before any Gate 3 model-benchmark
notebook is built. Gate 1's own exit criterion ("No target leakage possible by construction") is verified
live below, against the real data, not asserted from memory.

## Purpose
Produces BP1's Gate 1 output exactly as Section 8 defines it: a policy artifact recording the target
definition, leakage rules, and ASSUMPTIONs — verified against the real, profiled data (Sprint 1's
`01_data_acquisition_profiling.ipynb` and Gate 2's taxonomy-mapping notebook), not invented.

## Standing rules this notebook follows
- **Execution boundary** (Section 12.2): Claude wrote this notebook; it does not run it. You run it on
  your own machine, and the real, live-checked leakage/class-balance results below become this project's
  Gate 1 policy record.
- **Zero-fabrication** (Section 12.1): every check below runs against the real files in `data/external/`.
  The "no shared identifier column" claim is verified by actually comparing both real schemas, not stated
  from memory of what was documented earlier.
- **WARP**: `configure_performance()` first. BANKING77 is small (~13K rows) so this notebook loads it
  eagerly — no CFPB row-level data is loaded here at all (Gate 1 needs BANKING77's schema/split/labels
  only; CFPB's Product-level schema is compared structurally, not loaded).
- **HYPER**: reuses `src/taxonomy/taxonomy_mapper.CFPB_DTYPES` (for the CFPB column-name comparison) and
  `configs/taxonomy_mapping.yaml` (for the 9-bucket secondary target), rather than re-deriving either.
- **Idempotent**: re-running this notebook overwrites `configs/bp1_customer_intent_classification.yaml`
  and this notebook's own `policy.json` artifact in place.
- **PROJECT_STRUCTURE_LOCKED.md rule #3**: same resolver as every other notebook in this project.

## Business Understanding (Master Plan Section 10.2, BP1)
BP1 classifies real customer intent from free text into BANKING77's 77 real intent categories (and, for
cross-dataset reporting comparability only, the 9-bucket common taxonomy from Gate 2). Per
`docs/data_dictionary/RAW_DATA_MANIFEST.md` Finding 2, the CFPB extract used in this project has **no**
narrative/complaint-text column, so BP1's actual trainable text classifier is built and evaluated on
BANKING77 alone — CFPB's Gate-2-tagged rows feed BP8's aggregate/cross-dataset reporting, never this
classifier's training or evaluation data. This is stated as an explicit ASSUMPTION below, not left
implicit.

## Outputs (both written, idempotent overwrite-in-place)
- `configs/bp1_customer_intent_classification.yaml` — `target_definition`, `leakage_rules`, `status`
  updated in place (existing `assumptions` entries preserved, new ones appended)
- `notebooks/bp1_customer_intent_classification/artifacts/policy.json` — the Section 8 Gate 1 output
  artifact, with the live leakage-check and class-balance results embedded

## Prerequisites
`01_data_acquisition_profiling.ipynb` and the Gate 2 taxonomy-mapping notebook should both have been
real-run at least once (Sprint 1 / Sprint 2 ordering), though this notebook re-verifies what it needs
independently rather than trusting their artifacts blindly.

## If a structural check below fails
It raises `AssertionError` with the failing check named. A failing leakage check in particular must never
be worked around — if train/test text overlap is ever found to be nonzero, that is a real modeling risk
(inflated evaluation metrics) and must be fixed in the split, not in this check.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp1_customer_intent_classification/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402

from taxonomy.taxonomy_mapper import CFPB_DTYPES, load_mapping_config  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL_DIR / "banking77_test.csv"
B77_CATEGORIES_PATH = DATA_EXTERNAL_DIR / "banking77_categories.json"

# ============================================================
# SECTION 4: Structural leakage check #1 - no shared identifier/join column between CFPB and BANKING77
# (verified live against the real schemas, not asserted from memory)
# ============================================================
cfpb_columns = set(CFPB_DTYPES.keys())
banking77_columns = set(pl.read_csv(B77_TRAIN_PATH, n_rows=1).columns)
shared_columns = cfpb_columns & banking77_columns
print(f"[OK] CFPB columns: {sorted(cfpb_columns)}")
print(f"[OK] BANKING77 columns: {sorted(banking77_columns)}")
print(f"[OK] Shared column names between the two schemas: {sorted(shared_columns) or 'NONE'}")

# ============================================================
# SECTION 5: Structural leakage check #2 - zero exact-text overlap between BANKING77 train and test
# ============================================================
train = pl.read_csv(B77_TRAIN_PATH, dtypes={"text": pl.Utf8, "category": pl.Categorical})
test = pl.read_csv(B77_TEST_PATH, dtypes={"text": pl.Utf8, "category": pl.Categorical})

train_texts = set(train["text"].to_list())
test_texts = set(test["text"].to_list())
overlap_texts = train_texts & test_texts
print(f"[OK] BANKING77 train rows: {train.height:,}, test rows: {test.height:,}")
print(f"[OK] Exact-text overlap between train and test: {len(overlap_texts)} row(s)")

# ============================================================
# SECTION 6: Class balance - 77-class (BANKING77 category) and 9-class (common taxonomy bucket)
# ============================================================
with open(B77_CATEGORIES_PATH, "r", encoding="utf-8") as f:
    b77_categories = json.load(f)

train_class_counts = (
    train.group_by("category").agg(pl.len().alias("n")).sort("n", descending=True)
)
train_counts_list = train_class_counts["n"].to_list()
class_imbalance_77 = {
    "n_classes": len(b77_categories),
    "n_classes_in_train": train_class_counts.height,
    "min_class_count": int(min(train_counts_list)) if train_counts_list else None,
    "max_class_count": int(max(train_counts_list)) if train_counts_list else None,
    "imbalance_ratio_max_over_min": round(max(train_counts_list) / min(train_counts_list), 2)
    if train_counts_list and min(train_counts_list) > 0 else None,
}
print(f"[OK] 77-class balance (train split): {class_imbalance_77}")

taxonomy_config_path = CONFIGS_DIR / "taxonomy_mapping.yaml"
class_imbalance_9 = None
if taxonomy_config_path.exists():
    mapping = load_mapping_config(taxonomy_config_path)
    # Vectorized (WARP) - same pattern as taxonomy_mapper.load_banking77_with_bucket, never a per-row apply.
    train_with_bucket = train.with_columns(
        pl.col("category").cast(pl.Utf8)
        .replace(mapping["banking77_category_to_bucket"], default="UNMAPPED_UNKNOWN_CATEGORY")
        .alias("common_taxonomy_bucket")
    )
    bucket_counts = (
        train_with_bucket.group_by("common_taxonomy_bucket").agg(pl.len().alias("n")).sort("n", descending=True)
    )
    bucket_counts_list = bucket_counts["n"].to_list()
    class_imbalance_9 = {
        "n_buckets_in_train": bucket_counts.height,
        "min_bucket_count": int(min(bucket_counts_list)) if bucket_counts_list else None,
        "max_bucket_count": int(max(bucket_counts_list)) if bucket_counts_list else None,
        "imbalance_ratio_max_over_min": round(max(bucket_counts_list) / min(bucket_counts_list), 2)
        if bucket_counts_list and min(bucket_counts_list) > 0 else None,
    }
    print(f"[OK] 9-bucket balance (train split): {class_imbalance_9}")
else:
    print("[INFO] taxonomy_mapping.yaml not found - skipping 9-bucket balance (run Gate 2 first for this).")

# ============================================================
# SECTION 7: Assemble the Gate 1 policy (target definition, leakage rules, assumptions)
# ============================================================
policy = {
    "bp_id": "bp1",
    "bp_name": "bp1_customer_intent_classification",
    "gate": 1,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "target_definition": {
        "primary_target": "category",
        "primary_target_description": "BANKING77's real 77-class fine-grained customer intent label, "
                                       "attached to the `text` field it was collected with.",
        "secondary_target": "common_taxonomy_bucket",
        "secondary_target_description": "The 9-bucket common taxonomy from Gate 2 "
                                         "(configs/taxonomy_mapping.yaml), used only for coarser reporting "
                                         "and for cross-dataset comparability with CFPB's bucket-tagged "
                                         "rows in BP8 - never trained on directly in place of the real "
                                         "77-class label.",
        "feature_variable": "text",
        "train_test_split_source": "BANKING77's own provided train/test split (banking77_train.csv / "
                                    "banking77_test.csv) - used as-is, never re-randomized or re-merged.",
    },
    "leakage_rules": [
        "No CFPB row-level data is ever joined into BANKING77 training or evaluation data - verified live "
        "in Section 4: the two real schemas share zero column names, so no identifier-based join is even "
        "possible by construction.",
        "The BANKING77 train/test split is used exactly as provided by the source dataset and is never "
        "re-randomized, re-merged, or re-split - verified live in Section 5 (exact-text overlap check).",
        f"Live exact-text overlap between train and test at Gate 1 time: {len(overlap_texts)} row(s) - "
        "must be 0 for this policy to be considered valid; see the structural check below.",
        "CFPB's Gate-2 bucket-tagged rows are descriptive/reporting inputs only (BP8) and are never used "
        "as training or evaluation data for BP1's classifier.",
    ],
    "assumptions": [
        "CFPB<->BANKING77 integration is a taxonomy/semantic crosswalk (configs/taxonomy_mapping.yaml), "
        "not a row-level join - see docs/data_dictionary/CFPB_BANKING77_TAXONOMY_MAPPING.md.",
        "Only ~6.55% of the real CFPB extract (Checking/savings, Credit card/prepaid card, Money transfer "
        "products) has any plausible overlap with BANKING77's 77 intents; the rest is out of scope for "
        "this integration.",
        "BP1's trainable text classifier is built and evaluated on BANKING77 alone, because the CFPB "
        "extract used in this project has no narrative/complaint-text column "
        "(docs/data_dictionary/RAW_DATA_MANIFEST.md Finding 2).",
    ],
    "compliance_touchpoint": {
        "requirement": "Data-minimization & purpose-limitation statement (GLBA/GDPR-aligned)",
        "statement": "BP1 processes only the `text` and `category` fields of the publicly released "
                     "BANKING77 dataset (anonymized customer-service intent utterances, no real-customer "
                     "PII) for the stated purpose of intent classification. CFPB fields used elsewhere in "
                     "this suite are limited to structured Product/Sub-product/Issue categories already "
                     "published by CFPB - no narrative text, no demographic or protected-class field, and "
                     "no CFPB data is used as training or evaluation data for this classifier. No data is "
                     "collected, retained, or processed beyond what is documented here.",
    },
    "live_checks": {
        "shared_columns_cfpb_banking77": sorted(shared_columns),
        "train_test_exact_text_overlap_rows": len(overlap_texts),
        "class_imbalance_77_class": class_imbalance_77,
        "class_imbalance_9_bucket": class_imbalance_9,
    },
}

# ============================================================
# SECTION 8: Write outputs (idempotent overwrite-in-place)
# ============================================================
policy_json_path = ARTIFACTS_DIR / "policy.json"
with open(policy_json_path, "w", encoding="utf-8") as f:
    json.dump(policy, f, indent=2)
print(f"\n[SAVED] {policy_json_path.relative_to(PROJECT_ROOT)}")

bp1_config_path = CONFIGS_DIR / "bp1_customer_intent_classification.yaml"

# Gate 1 owns ONLY the front-matter section of this shared config file (bp_id through
# random_state) - Gates 2-5 each own exactly one marker-delimited block below it. A prior
# version of this cell did a blind full-file overwrite here, which silently destroyed Gates
# 3/4/5's already-recorded blocks whenever Gate 1 was re-run after them (real incident,
# LESSONS_LEARNED_APPLIED.md #20). write_front_matter() replaces only this section and
# preserves every existing gate block verbatim, regardless of position or order.
from utils.bp1_config_sync import read_existing_gate_block_markers, write_front_matter  # noqa: E402

_existing_gate_markers = read_existing_gate_block_markers(bp1_config_path)
_status_suffix = ""
for _gate_num, _gate_label in ((3, "Gate 3"), (4, "Gate 4"), (5, "Gate 5")):
    if any(_gate_label in _m for _m in _existing_gate_markers):
        _status_suffix += f"_gate{_gate_num}_confirmed"

bp1_config_text = f"""# Per-BP config - filled in at Gate 1 (Business Understanding & Policy)
# Gate 1 owns bp_id through random_state below via write_front_matter() (src/utils/bp1_config_sync.py);
# Gates 2-5 each own exactly one marker-delimited block appended after it via write_gate_block() -
# do not hand-edit either section, re-run the owning notebook instead.
bp_id: "bp1"
bp_name: "bp1_customer_intent_classification"
status: "gate1_confirmed_gate2_confirmed{_status_suffix}"   # not_started | gate1 | gate2_in_progress | gate2 | gate3 | gate4 | gate5 | gate6_complete
target_definition:
  primary_target: "category"
  secondary_target: "common_taxonomy_bucket"
  feature_variable: "text"
  train_test_split_source: "BANKING77's own provided train/test split - never re-randomized."
leakage_rules:
  - "No CFPB row-level data is ever joined into BANKING77 training/evaluation data (zero shared columns,
     verified live in the Gate 1 notebook)."
  - "BANKING77 train/test split used as provided, never re-randomized or re-merged (live exact-text
     overlap check run every Gate 1 execution)."
  - "CFPB Gate-2 bucket-tagged rows are descriptive/reporting inputs only (BP8), never training/eval data
     for this classifier."
assumptions:
  - "CFPB<->BANKING77 integration is a taxonomy/semantic crosswalk (configs/taxonomy_mapping.yaml), not a
     row-level join - see docs/data_dictionary/CFPB_BANKING77_TAXONOMY_MAPPING.md."
  - "Only ~6.55% of the real CFPB extract (Checking/savings, Credit card/prepaid card, Money transfer products)
     has any plausible overlap with BANKING77's 77 intents; the rest is out of scope for this integration."
  - "BP1's trainable text classifier is built and evaluated on BANKING77 alone, because the CFPB extract used
     in this project has no narrative/complaint-text column (RAW_DATA_MANIFEST.md Finding 2)."
random_state: 42
"""
write_front_matter(bp1_config_path, bp1_config_text)
print(f"[SAVED] {bp1_config_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "no_shared_identifier_column_cfpb_banking77": len(shared_columns) == 0,
    "banking77_train_test_zero_exact_text_overlap": len(overlap_texts) == 0,
    "all_77_categories_known": len(b77_categories) == 77,
    "class_imbalance_computed": class_imbalance_77.get("min_class_count") is not None,
    "policy_json_written": policy_json_path.exists(),
    "bp1_config_yaml_written": bp1_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print("\n[ALL CHECKS PASSED] BP1 Gate 1 complete - target defined, leakage rules verified live, "
      "ASSUMPTIONs recorded. Proceed to BP1 Gate 3 (Model / Classifier Benchmark) next.")
